Day 19: Time-Based Retention Analysis

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("meetmux_transactions.csv")
df['PurchaseDate'] = pd.to_datetime(df['PurchaseDate'])

df['OrderMonth'] = df['PurchaseDate'].dt.to_period('M')

df['CohortMonth'] = df.groupby('CustomerID')['PurchaseDate'] \
                      .transform('min') \
                      .dt.to_period('M')

def get_date_int(df, column):
    year = df[column].dt.year
    month = df[column].dt.month
    return year, month

order_year, order_month = get_date_int(df, 'OrderMonth')
cohort_year, cohort_month = get_date_int(df, 'CohortMonth')

df['CohortIndex'] = (order_year - cohort_year) * 12 + (order_month - cohort_month) + 1

cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'] \
                .nunique() \
                .reset_index()

cohort_pivot = cohort_data.pivot(index='CohortMonth',
                                columns='CohortIndex',
                                values='CustomerID')

cohort_sizes = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_sizes, axis=0)

plt.figure(figsize=(12, 8))

sns.heatmap(retention,
            annot=True,
            fmt='.0%',
            cmap='YlGnBu',
            linewidths=0.5)

plt.title('MeetMux User Retention Cohorts')
plt.xlabel('Months Since First Purchase')
plt.ylabel('Cohort Month')

plt.show()

avg_retention = retention.mean()
retention_drop = avg_retention.diff()

cliff_month = retention_drop.idxmin()

print("Average Retention by Month:\n", avg_retention)
print("\nDrop Between Months:\n", retention_drop)
print(f"\n⚠️ Biggest drop occurs at Month {cliff_month}")

month_3_retention = retention[3].mean()
print(f"\n📊 Average Month 3 Retention: {month_3_retention:.2%}")

Reflection:
The January cohort having 40% retention and March only 15% shows that March users were less engaged.
This suggests either:
1:Lower-quality users were acquired in March, or
2:Changes in the app negatively affected user experience

Conclusion: Something changed between January and March that reduced retention, and it should be investigated.